# Taller LSTM: demanda eléctrica de la siguiente hora
**Anderson David Ojeda Zambrano · UCC Pasto**

Referencia reconocida: [samumafu/Ia-Taller-ltsm](https://github.com/samumafu/Ia-Taller-ltsm). Implementación y resultados propios. Extrae el ZIP completo antes de ejecutar. Consulta el informe PDF para las nueve respuestas y resultados de la ejecución entregada.

## 1. Entorno
Ejecuta las celdas en orden desde esta carpeta. Si faltan dependencias, descomenta la instalación. La ejecución de entrenamiento regenera resultados y puede tardar varios minutos.

In [ ]:
# %pip install -r requirements.txt
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Image
assert Path("pipeline.py").exists(), "Entra a la carpeta taller_lstm_anderson antes de continuar."

## 2. Explorar el dataset
Buscamos faltantes, duplicados, desorden y valores fuera de límites físicos. Un valor extremo no demuestra por sí solo un error.

In [ ]:
from pipeline import RAW, prepare, windows
raw = pd.read_csv(RAW)
print(raw.shape)
display(raw.head())
display(raw.isna().sum())
display(raw.describe().T)

## 3. Limpieza y prevención de fuga
No se interpolan valores. Se conserva la rejilla horaria y se descartan ventanas incompletas. La etiqueta de la hora k debe coincidir con la declarada en k-1. Los 40 desacuerdos se auditan y no se usan como etiquetas. La demanda objetivo nunca entra en X. La división se hace por la fecha del objetivo, no por la última entrada.

In [ ]:
d, x, y, grupos, scaler, auditoria = prepare(raw)
print(json.dumps(auditoria, indent=2, ensure_ascii=False))
for w in [12,24,48]:
    print(w, {g: windows(x, ids, w).shape for g, ids in grupos.items()})

## 4. Diseñar tres arquitecturas
**Base:** LSTM(32) → Dense(1).

**Profunda:** LSTM(64) → Dropout → LSTM(32) → Dropout → Dense(1).

**Propuesta:** Conv1D causal → LSTM(32) → Dropout → Dense(16) → Dense(1). La convolución busca patrones locales y la LSTM dependencias temporales.

In [ ]:
from train import build
for nombre in ["base", "profunda", "propuesta"]:
    build(nombre, 24, x.shape[1]).summary()

## 5. Entrenar los nueve experimentos
EarlyStopping observa validación y restaura los mejores pesos. Se fija la selección por RMSE de validación antes de evaluar prueba. `REENTRENAR=False` permite consultar la ejecución real incluida sin repetirla; cámbialo a `True` para reproducirla.

In [ ]:
REENTRENAR = False
if REENTRENAR:
    import subprocess
    subprocess.run([sys.executable, "train.py", "--epochs", "40", "--patience", "7"], check=True)
assert Path("results/finalizado.json").exists(), "Ejecuta el entrenamiento primero."
tabla = pd.read_csv("results/resultados_modelos.csv")
display(tabla[["Modelo","Ventana","MAE","MSE","RMSE","MAPE","R2","Epocas","Seleccionado"]])
display(pd.read_csv("results/baselines.csv"))

## 6. Gráficas y generalización
Compara loss de entrenamiento/validación, error y demanda predicha. Los gráficos muestran MW originales; loss utiliza el objetivo estandarizado. La tabla contiene solo las ventanas válidas comunes.

In [ ]:
display(Image(filename="results/comparacion.png"))
cfg=json.loads(Path("models/config.json").read_text())
tag=f'{cfg["model"]}_v{cfg["window"]}'
display(Image(filename=f"results/loss_{tag}.png"))
display(Image(filename=f"results/prediccion_error_{tag}.png"))
display(tabla[["Modelo","Ventana","Train_RMSE","Val_RMSE","RMSE"]])

## 7. Variables influyentes
La perturbación por bloques se calcula únicamente en validación. Es exploratoria y no demuestra causalidad: las variables correlacionadas pueden sustituirse y la perturbación puede crear combinaciones poco realistas.

In [ ]:
display(pd.read_csv("results/importancia_validacion.csv"))

## 8. Inferencia y app
La entrada es una secuencia histórica de n filas, no una sola fila repetida. La hora objetivo es posterior a la última entrada.

In [ ]:
from tensorflow import keras
from pipeline import inference
modelo=keras.models.load_model("models/mejor_modelo.keras", compile=False)
ejemplo=pd.read_csv("data/ejemplo_entrada.csv")
prediccion,hora=inference(ejemplo,modelo,cfg)
print(f"Demanda para {hora}: {prediccion:.2f} MW")

## 9. Pruebas y entrega
En terminal: `python -m unittest discover -s . -p test_pipeline.py -v` y `python -m streamlit run app.py`.

Lee `Informe_Taller_LSTM_Anderson.pdf` para las respuestas a las nueve preguntas. El despliegue público se completa con los pasos del README y la URL resultante. El paquete no afirma tener una URL pública activa.

Fuentes oficiales: [LSTM](https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM), [EarlyStopping](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping), [Streamlit](https://docs.streamlit.io/deploy/streamlit-community-cloud/deploy-your-app).